# Project 16 — Detecting a shift (single change-point)

**Scenario.** A reaction-kinetics time series of **counts** undergoes a **regime shift** at an unknown time $\tau$ — a catalyst is added, a pathway switches on — after which the underlying Poisson rate changes. We want to locate the shift and quantify the before/after rates.

**New skill.** Discrete *structural* change. **Key pitfall.** A **multimodal posterior over $\tau$** — if more than one time looks like a plausible shift, the $\tau$ posterior has multiple peaks and its *mean* is meaningless. We show two implementations (discrete $\tau$ and a marginalized $\tau$) and how to summarize a multimodal $\tau$.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

$\text{rate}_t=\lambda_0$ if $t<\tau$ else $\lambda_1$; $y_t\sim\text{Poisson}(\text{rate}_t)$. **Index convention:** $\tau$ is the **first post-shift index**, so $0..\tau{-}1$ use $\lambda_0$ and $\tau..T{-}1$ use $\lambda_1$. **Assumptions:** (a) exactly one shift, (b) abrupt (not gradual) change, (c) constant rate within each regime, (d) Poisson counts (mean = variance). Truth: $\tau=70, \lambda_0=4, \lambda_1=11, T=120$.

In [ ]:
from data.generate_data import generate
data = generate(); y = data['y']; t = data['truth']
print(f"T={data['T']}, true tau={t['tau']}, lam0={t['lam0']}, lam1={t['lam1']}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
ax.bar(range(len(y)), y, color='#4C72B0', width=1.0)
ax.axvline(t['tau']-0.5, color='red', ls='--', label='true tau')
ax.set(xlabel='time t', ylabel='count y_t', title='Count series with a regime shift')
ax.legend(); plt.tight_layout()

## Step 2 — Model: discrete tau (the classic switch model)

$$\tau\sim\text{DiscreteUniform}(1, T{-}1),\quad \lambda_0,\lambda_1\sim\text{Exponential}(1/5),\quad \text{rate}_t=\text{switch}(\tau>t,\ \lambda_0,\ \lambda_1).$$

PyMC samples $\lambda$ with NUTS and the integer $\tau$ with Metropolis. The `switch(tau > t, lam0, lam1)` direction is the easy place to introduce an off-by-one or sign error (see the broken notebook).

In [ ]:
from model import build_model, fit, build_marginal_model, fit_marginal, tau_posterior_from_marginal
model = build_model(data); model

## Step 3 — Prior predictive checks

The prior on the rates is Exponential(1/5) (mean 5). Prior-predictive count series should span a plausible range of levels and shift locations — not, say, rates in the thousands. We check the implied count magnitudes are sane.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=300, random_seed=RNG)
pp = prior.prior_predictive['y'].values.reshape(-1, len(y))
print('prior-predictive count range:', int(pp.min()), 'to', int(pp.max()),
      '| median per-series mean:', round(float(np.median(pp.mean(1))),2))

## Step 4 — Inference (NUTS + Metropolis)

`draws=1000, tune=1000, chains=4`. Four chains help confirm the $\tau$ posterior is the same mode across chains (a multimodal $\tau$ would show chains favouring different peaks).

In [ ]:
idata = fit(data, draws=1000, tune=1000, chains=4, seed=16)

## Step 5 — Diagnostics & the tau posterior

Check R-hat/ESS for the rates. For $\tau$, **do not summarize by the mean** — inspect the full discrete posterior. Here the shift is sharp so $\tau$ concentrates on one value; we still plot the distribution and report the **mode** and a credible set.

In [ ]:
print(az.summary(idata, var_names=['lam0','lam1']))
tau_draws = idata.posterior['tau'].values.ravel().astype(int)
vals, counts = np.unique(tau_draws, return_counts=True)
mode = vals[counts.argmax()]
print(f'tau MODE = {mode} (true {t["tau"]}); tau MEAN = {tau_draws.mean():.2f} '
      f'(mean can mislead if multimodal)')

In [ ]:
fig, ax = plt.subplots(figsize=(7,3))
ax.bar(vals, counts/counts.sum(), color='#55A868', width=1.0)
ax.axvline(t['tau'], color='red', ls='--', label='true tau')
ax.set(xlabel='tau', ylabel='P(tau | y)', title='Posterior over the change-point')
ax.legend(); plt.tight_layout()

## Step 6 — Posterior predictive checks

Compare observed counts to posterior-predictive counts; the two-level structure (low before, high after) should be reproduced.

In [ ]:
ax = az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

## Step 7 — A second implementation: marginalize tau

We can sum the likelihood over all $T{-}1$ candidate $\tau$ analytically, leaving only $\lambda_0,\lambda_1$ to sample (pure NUTS, no discrete step). The full $P(\tau\mid y)$ is then reconstructed from the per-$\tau$ weights. The two implementations should agree — a strong cross-check.

In [ ]:
idm = fit_marginal(data, draws=1000, tune=1000, chains=2, seed=16)
print(az.summary(idm, var_names=['lam0','lam1']))
probs = tau_posterior_from_marginal(idm, data)
print('marginal-model tau MODE =', int(np.argmax(probs))+1, '(true', t['tau'], ')')

In [ ]:
fig, ax = plt.subplots(figsize=(7,3))
ax.bar(np.arange(1, data['T']), probs, color='#8172B3', width=1.0)
ax.axvline(t['tau'], color='red', ls='--', label='true tau')
ax.set(xlabel='tau', ylabel='P(tau | y)', title='Marginalized model: P(tau | y)')
ax.legend(); plt.tight_layout()

## Step 8 — Decision & communication

For a collaborator: 'The rate jumps from ~4 to ~11 counts/bin at about bin 70 (94% credible set [68, 71]).' Report the **mode/credible set** of $\tau$, the two rates, and the **fold-change** $\lambda_1/\lambda_0$. See `summary_onepager.md`.

In [ ]:
l0 = idata.posterior['lam0'].values.ravel(); l1 = idata.posterior['lam1'].values.ravel()
fold = l1 / l0
print(f'rate before = {l0.mean():.2f}, after = {l1.mean():.2f}')
print(f'fold-change = {fold.mean():.2f} (94% [{np.percentile(fold,3):.2f}, {np.percentile(fold,97):.2f}])')